# Well-Architected, Cost & AZ-104 Exam Prep

Thirteen notebooks in, you have the Azure service map. This one ties the threads together. It walks the **Microsoft Azure Well-Architected Framework** — the same five pillars that the platform's official reviews use — and shows how the services you have already met line up under each one. It covers **Cost Management** as the operational discipline that turns architectural choices into a controlled monthly bill. And it closes with a study plan for **AZ-104 (Azure Administrator)**: which notebooks map to which exam-blueprint sections, how to read a scenario question, and the distractor patterns that show up over and over.

There is nothing new in this notebook — every service mentioned has been built in the previous thirteen. The work here is *composition*: which combinations satisfy which non-functional requirements, and which trade-offs you accept when you pick them.

## The Well-Architected Framework — five pillars

Microsoft's **Well-Architected Framework (WAF)** is the structured way the field talks about "is this architecture good?" Five pillars, each with its own checklist, scoring tool (in the Azure portal), and design principles:

- **Reliability** — does the workload meet its uptime and recovery targets?
- **Security** — does it protect data and identities against credible threats?
- **Cost Optimization** — does it deliver value at the right cost?
- **Operational Excellence** — can you ship changes and respond to incidents safely?
- **Performance Efficiency** — does it scale to demand without overspending?

The pillars are not independent. Tightening security often costs more (Private Link, Defender plans, dedicated HSM). Improving reliability usually trades against cost (more zones, more regions). The WAF review process exists precisely to make those trade-offs explicit per workload, so nobody silently optimises for one pillar at the expense of another.

Below we walk each pillar with the concrete service-level checklist that emerges from the previous notebooks.

## Reliability — the concrete checklist

- **Region & zone strategy** — zone-redundant by default for production tiers (App Service Premium v3, VMSS Flexible, App Gateway v2, Standard Load Balancer, ZRS storage); multi-region active-passive or active-active for tiers whose RTO/RPO demands it.
- **Statelessness in compute** — App Service / Container Apps / AKS scale horizontally; sessions in Redis, files in Blob or Files, secrets in Key Vault, *not* on local disk.
- **Data-tier replication** — Azure SQL auto-failover groups, PostgreSQL Flexible Server zone-redundant HA + cross-region replicas, Cosmos DB multi-region writes, GZRS storage.
- **Health probes and circuit breakers** — Load Balancer / App Gateway / Front Door probes hit *real* health endpoints (DB reachable, dependency reachable), not just TCP 200.
- **Backup and restore drills** — Azure Backup with immutable vault; quarterly restore tests; documented RTO/RPO per workload.
- **DR exercises** — ASR test failover quarterly; chaos-style game days on critical workloads.

## Security — the concrete checklist

- **Identity** — Entra ID with Conditional Access for all admin paths; PIM for privileged role activation; no permanent Owners on production.
- **Workload identity** — managed identities everywhere code calls Azure; OIDC federation from pipelines; no client secrets in CI.
- **Secrets** — Key Vault (RBAC mode, soft delete + purge protection); auto-rotation policies on keys; secrets that *must* exist are pulled at runtime, not baked into images.
- **Network** — Private Link endpoints on PaaS that supports it; storage and Key Vault firewalls denying public; Azure Firewall / Front Door WAF on egress and ingress respectively.
- **Defender plans** — for Servers, Containers, SQL, Storage, Key Vault — anywhere data sits or workloads run.
- **Sentinel** — for multi-subscription, multi-cloud, or SOC-driven environments; analytics rules tuned; playbooks automated for high-severity incidents.
- **Compliance** — regulatory standards mapped in Defender for Cloud's Compliance dashboard; policy initiatives enforcing the controls (Microsoft Cloud Security Benchmark at minimum).

## Cost Optimization — the concrete checklist

- **Right-size compute** — Advisor's right-sizing recommendations reviewed weekly; v3/v5 generations preferred over older; AMD `a` variants where supported.
- **Commit on the steady-state base** — Reserved Instances or Savings Plans for predictable load (typically 60–70% of fleet).
- **Spot for the burst layer** — batch, CI agents, stateless workers can take ~90% discount and 30-second eviction notice.
- **Hybrid Benefit** for Windows / SQL licensing where you have on-prem SA.
- **Storage tiers and lifecycle** — Hot → Cool → Cold → Archive lifecycle on blob; Standard SSD or HDD for non-production disks; Premium SSD v2 over classic Premium to decouple IOPS from size.
- **Scale to zero where supported** — Container Apps, Functions Consumption/Flex, SQL Serverless.
- **Stop dev/test outside hours** — Auto-shutdown on VMs; pause Synapse dedicated pools.
- **Govern via tags** — every resource has `environment`, `cost-center`, `application`, `owner`; Cost Management groups by tag for chargeback.

## Operational Excellence — the concrete checklist

- **Everything is in IaC** — Bicep modules per workload; landing zones via Cloud Adoption Framework templates; manual portal changes flagged and reverted.
- **Pipelines deploy via OIDC** — short-lived tokens, least-privilege role assignments per environment.
- **Safe deployment** — slot swaps / revisions / rolling upgrades with health gates; What-if diff on every PR.
- **Observability is on by default** — diagnostic settings policy with DeployIfNotExists; one Log Analytics workspace per environment; App Insights workspace-based with auto-instrumentation.
- **Alerts wired to action groups** with severity-aligned routing; smart detection on every App Insights instance.
- **Runbooks as Workbooks**, not as Word documents.
- **Service Health alerts** for the regions you live in.
- **Drills** — quarterly DR, monthly restore, ad-hoc chaos.

## Performance Efficiency — the concrete checklist

- **Pick the right compute surface** — Functions for sporadic event work, App Service for classic web, Container Apps for scale-to-zero microservices, AKS only when raw Kubernetes is needed (notebook 04 decision tree).
- **Cache aggressively** — Redis Premium for hot data; Front Door caching at the edge; CDN-style for static assets.
- **Asynchronous patterns** — Service Bus / Event Grid / Event Hubs decouple latency-sensitive paths from slow downstream work.
- **Pick the right data engine per access pattern** — Azure SQL for OLTP, Cosmos for globally-distributed JSON, Synapse / Fabric for analytics, ADLS Gen2 for the lake, Cognitive Search for full text.
- **Partition right** — Cosmos partition keys with high cardinality and even access; Event Hubs partition counts sized for peak throughput, not average.
- **Autoscale rules with cooldowns** — never flap; scale-out aggressive, scale-in conservative.
- **Premium SSD v2 / Ultra** — for workloads whose disk is the bottleneck; accelerated networking on every VM.

## Cost Management & Billing

**Microsoft Cost Management** is the toolset for understanding, forecasting, and controlling Azure spend. Three core surfaces:

- **Cost Analysis** — interactive charts of spend grouped by subscription, resource group, service, region, tag. The default workspace for monthly cost reviews.
- **Budgets** — alert thresholds (50%, 80%, 100% of monthly target) that fire to action groups or Logic Apps. Cannot stop spend by themselves — they notify.
- **Exports** — schedule a CSV / parquet export of cost details to a storage account for BI / data-warehouse ingestion. The right pattern for chargeback at scale.

**Scopes** — Cost Management understands the same hierarchy you saw in notebook 01: billing account → billing profile / EA enrollment → subscription → resource group → tag dimension. Permissions cascade; tags pivot the views.

**Pricing Calculator** estimates cost before deployment — punch in the SKUs and get a monthly figure. **TCO Calculator** compares running on-prem versus on Azure for migration business cases.

**Reservations, Savings Plans, Hybrid Benefit** — covered in notebook 03; this is where they show up financially. The Cost Analysis view has built-in *purchase recommendations* — Microsoft looks at your last 30/60/90 days of utilisation and suggests the RIs / Savings Plans that would have saved the most. Treat them as a starting point, not gospel.

AWS comparison: Cost Management ≈ AWS Cost Explorer + Budgets + Cost & Usage Reports; Pricing Calculator ≈ AWS Pricing Calculator; TCO ≈ AWS Migration Evaluator.

## AZ-104 blueprint — where each notebook fits

The **AZ-104 (Azure Administrator)** exam tests operational competence across the platform. The blueprint clusters into five areas; here is the mapping to this series:

| AZ-104 area | Notebooks |
|---|---|
| Manage Entra ID identities | **02** (Entra ID, RBAC, Policy, MGs) |
| Manage governance & subscriptions | **01** (hierarchy, regions), **02** (Policy, locks, tags), **12** (Activity Log) |
| Implement & manage storage | **05** (accounts, redundancy, blobs, Files, lifecycle, SAS) |
| Deploy & manage compute | **03** (VMs, VMSS, availability), **04** (App Service, Functions, containers) |
| Configure virtual networking | **06** (VNets, NSGs, routes, peering, VPN/ER), **07** (LB, App Gateway, Front Door, DNS) |
| Monitor & maintain Azure | **12** (Monitor, Log Analytics, App Insights, alerts), **13** (Backup, ASR) |

Notebooks **08** (databases), **09** (NoSQL/analytics), **10** (messaging), **11** (security services beyond Entra), and **14** (this one) extend beyond the AZ-104 blueprint into AZ-204 (developer), AZ-305 (architect), AZ-500 (security), DP-203 (data engineer) territory. Studying them anyway gives you the architectural context that turns brittle exam knowledge into durable platform understanding.

## Reading an Azure scenario question

AZ-104 (and most Microsoft cert exams) lean on multi-sentence scenarios with a constraint and a goal. The pattern that turns reading time into accuracy:

1. **Find the verb first.** "You need to **ensure** / **provide** / **prevent** ..." tells you the goal. Skim past everything that doesn't shape the answer until you've found it.
2. **Highlight the constraints** — *least privilege*, *with minimum effort*, *no downtime*, *cost-effective*, *no public IP*. Each constraint eliminates options.
3. **Match the constraint to a service**, not to a feature. "Globally available, edge-cached, with WAF" → Front Door, not App Gateway. "Reliable message processing with order" → Service Bus, not Event Grid.
4. **Eliminate by capability mismatch.** If an option simply cannot do what's asked (Basic Load Balancer has no zone redundancy; Cosmos DB free tier does not support multi-region), strike it. Two-of-four eliminations are usually possible from capability alone.
5. **Decide between the survivors on the constraint.** *Minimum effort* → managed service over IaaS. *Cost-effective* → Standard tier over Premium unless Premium-only feature mentioned. *No downtime* → blue-green or canary, not in-place.

Most scenarios reward recognising *which Azure noun the question describes*. If the right noun is in your head — and the previous notebooks put it there — the question becomes a translation exercise rather than a guess.

## Distractor patterns to recognise

A few recurring traps:

- **Basic SKUs in production answers** — Basic Load Balancer, Basic Public IP, Basic Cosmos. Mostly there to be eliminated.
- **Shared key when managed identity is available** — questions that mention "least-privileged" and offer both a connection string and a managed identity. Always managed identity.
- **Access Policy vs RBAC on Key Vault** — modern answers use RBAC.
- **Service Endpoint vs Private Endpoint** — modern answers use Private Endpoint.
- **NSG vs Azure Firewall** — NSGs handle subnet-level filtering by 5-tuple; Firewall handles centralised L3–L7 with FQDN rules. Pick the layer the scenario describes.
- **Availability Set vs Availability Zone vs Region** — "survive a datacenter failure" → zones; "survive a rack failure in a region without zones" → availability set; "survive a region failure" → multi-region.
- **Service Bus vs Event Grid vs Event Hubs** — work queue → Service Bus; event notification → Event Grid; high-volume stream → Event Hubs.
- **Blueprints in 2026** — deprecated; modern answer is Deployment Stacks + policy.
- **AAD** vs **Entra ID** — same product, the name changed. Microsoft is updating exam wording.

When a question offers a tempting old-school answer alongside a newer managed primitive, the newer one is almost always correct on current exams.

## Study plan

A practical four-week run at AZ-104 once you have read this series:

- **Week 1 — Foundations and identity** — re-read notebooks 01, 02, 06. Build a lab tenant: management group tree, two subscriptions, an Entra ID test user, Conditional Access policy, RBAC assignments at a resource group, Azure Policy denying public IPs. Spin up a small VNet with NSGs and a peering.
- **Week 2 — Compute and storage** — notebooks 03, 04, 05. Deploy a VM Scale Set across zones with autoscale. Deploy an App Service with a slot swap. Build a storage account with HNS, lifecycle, soft delete, and a private endpoint.
- **Week 3 — Networking, traffic, and monitor** — notebooks 06, 07, 12. Add an Application Gateway in front of the App Service. Stand up a Log Analytics workspace, diagnostic settings, a metric alert, an action group. Practise KQL queries against the App Service request logs.
- **Week 4 — Drills and gaps** — notebook 13 for Backup and ASR. Run the Microsoft Learn AZ-104 practice assessment. Go back and re-read any notebook you missed more than 20% of questions from.

On exam day: read the verb first, eliminate Basic SKUs and legacy patterns, prefer managed identities, prefer Private Link, and trust the picking rule for the three messengers. The platform underneath is the same one you have just learned — the exam is asking you to name the right primitive on demand.

## Closing

Fourteen notebooks in, the shape of the platform should be steady in your head: a four-level hierarchy controlled by one API, identity and policy enforced at the chokepoint, regions and zones as the physical substrate, a portfolio of compute / storage / network / data / integration / security services that each solve a specific problem at a specific layer, and the operational disciplines — observability, deployment, backup, DR, cost — that turn the parts into a running system.

Azure is wide but it composes. Every workload you will design from here uses the same primitives in different combinations. The work is no longer learning what the pieces are; it is recognising which combination fits the requirement in front of you. That is the muscle the rest of your career builds.

Good luck with the exam, and more importantly, with the systems you go on to build.